# E14 - Playground separado: BPE vs WordPiece

Este notebook esta armado para probar tokenizacion en vivo de manera ordenada:

1. Primero editas `entradas`.
2. Despues corres solo la seccion **A. Tokenizar con BPE**.
3. Luego corres solo la seccion **B. Tokenizar con WordPiece**.
4. Al final corres **C. Comparacion lado a lado**.

La idea docente es que el alumno vea la diferencia por etapas, no todo mezclado desde el inicio.


## Imagen guia del tema <!-- data-aem4-visual-assets -->

![e14 bpe en vivo](assets/visuals/e14_bpe_en_vivo.svg)

![e14 wordpiece en vivo](assets/visuals/e14_wordpiece_en_vivo.svg)

Usa esta imagen como punto de partida antes de ejecutar las celdas. La idea es que el alumno vea el flujo completo antes de mirar numeros.


## 0. Frases y palabras para probar

Edita esta lista durante la clase. Conviene mezclar ejemplos cotidianos, medicos, legales y financieros para que se vea cuando una palabra se parte mucho.

**Ejemplos para probar en vivo:**

| Tipo | Entrada | Que mirar |
|---|---|---|
| Cotidiana | `banco plaza` | pocas piezas y sentido simple |
| Financiera | `interchange fee capping` | terminos en ingles y dominio financiero |
| Medica | `hypercholesterolemia` | palabra larga y tecnica |
| Legal | `postcontractual confidentiality` | compuestos y vocabulario de dominio |
| Compliance | `AML thresholding` | siglas y termino tecnico |


In [ ]:
# Editar esta lista en vivo.
entradas = [
    "banco plaza",
    "financialization",
    "hypercholesterolemia",
    "interchange fee capping",
    "postcontractual confidentiality",
    "AML thresholding",
]


## 1. Utilidades didacticas

Estas funciones no son tokenizers reales de produccion. Son simuladores simples para explicar la intuicion:

- **BPE didactico:** busca la pieza mas larga conocida y la va consumiendo.
- **WordPiece didactico:** usa una segmentacion parecida, pero marca continuaciones con `##`.

En clase conviene decir: "Esto no reemplaza el tokenizer real de un modelo, pero nos permite ver la idea".


In [ ]:
PIEZAS = [
    "banco", "plaza", "inter", "change", "fee", "cap", "capping",
    "financial", "ization", "finance", "hyper", "cholesterol", "emia",
    "post", "contract", "contractual", "confidential", "confidentiality",
    "AML", "threshold", "thresholding", "settlement", "compliance", "real", "time",
]


def bpe_didactico(palabra):
    original = palabra
    lower = palabra.lower()
    salida = []

    while lower:
        candidatos = sorted(
            [p for p in PIEZAS if lower.startswith(p.lower())],
            key=len,
            reverse=True,
        )
        if candidatos:
            pieza = candidatos[0]
            salida.append(original[:len(pieza)])
            original = original[len(pieza):]
            lower = lower[len(pieza):]
        else:
            salida.append(original[0])
            original = original[1:]
            lower = lower[1:]

    return salida


def wordpiece_didactico(palabra):
    piezas = bpe_didactico(palabra)
    if not piezas:
        return []
    return [piezas[0]] + ["##" + p for p in piezas[1:]]


def tokenizar_bpe(texto):
    tokens = []
    for palabra in texto.split():
        tokens.extend(bpe_didactico(palabra))
    return tokens


def tokenizar_wordpiece(texto):
    tokens = []
    for palabra in texto.split():
        tokens.extend(wordpiece_didactico(palabra))
    return tokens


def barra(tokens):
    return "#" * len(tokens)


## A. Tokenizar solo con BPE

**Que mostrar:** BPE intenta construir tokens fusionando piezas frecuentes. En esta version didactica, busca piezas largas conocidas.

**Frase para decir:** "BPE intenta representar palabras nuevas usando pedazos que ya conoce. Si conoce `financial` e `ization`, puede armar `financialization` sin mandarla a `[UNK]`."


In [ ]:
print("TOKENIZACION BPE DIDACTICA")
print("=" * 32)

for entrada in entradas:
    tokens = tokenizar_bpe(entrada)
    print()
    print(f"Entrada: {entrada}")
    print(f"BPE tokens ({len(tokens)}): {tokens}")
    print(f"Visual: {barra(tokens)}")


### Preguntas despues de BPE

1. Que entrada se partio mas?
2. Que piezas todavia conservan significado?
3. Que pasa cuando aparece una palabra que no esta bien cubierta por `PIEZAS`?


## B. Tokenizar solo con WordPiece

**Que mostrar:** WordPiece tambien trabaja con subwords, pero suele marcar las continuaciones de palabra con `##`.

**Frase para decir:** "El prefijo `##` nos ayuda a ver que esa pieza no arranca una palabra nueva, sino que continua una palabra anterior."


In [ ]:
print("TOKENIZACION WORDPIECE DIDACTICA")
print("=" * 39)

for entrada in entradas:
    tokens = tokenizar_wordpiece(entrada)
    print()
    print(f"Entrada: {entrada}")
    print(f"WordPiece tokens ({len(tokens)}): {tokens}")
    print(f"Visual: {barra(tokens)}")


### Preguntas despues de WordPiece

1. Donde aparecen tokens con `##`?
2. Que indica que una pieza sea continuacion?
3. Que diferencia visual aparece frente a BPE?


## C. Comparacion lado a lado

Ahora si mostramos ambos metodos juntos. Esta celda sirve para cerrar la explicacion y discutir trade-offs.


In [ ]:
print("| entrada | BPE | # BPE | WordPiece | # WP | Diferencia visible |")
print("|---|---|---:|---|---:|---|")

for entrada in entradas:
    bpe = tokenizar_bpe(entrada)
    wp = tokenizar_wordpiece(entrada)
    diferencia = "misma cantidad"
    if bpe != wp:
        diferencia = "WordPiece marca continuaciones con ##"
    if len(bpe) != len(wp):
        diferencia = "cambia cantidad de piezas"
    print(f"| {entrada} | `{bpe}` | {len(bpe)} | `{wp}` | {len(wp)} | {diferencia} |")


In [ ]:
# Vista tipo barras para explicar costo aproximado por cantidad de tokens.
print("COMPARACION VISUAL")
print("=" * 20)
for entrada in entradas:
    bpe = tokenizar_bpe(entrada)
    wp = tokenizar_wordpiece(entrada)
    print(f"{entrada:<35} BPE {len(bpe):>2} {barra(bpe)} | WP {len(wp):>2} {barra(wp)}")


## D. Como probarlo en vivo

1. Agregar una palabra inventada, por ejemplo `superfinancialregulation`.
2. Agregar un termino medico largo.
3. Agregar una frase corta cotidiana.
4. Correr primero BPE, despues WordPiece y recien al final la comparacion.

## Mini desafio para alumnos

Pedirles que propongan tres terminos de su dominio y anticipen antes de ejecutar:

- se parte mucho o poco?
- que piezas conservarian significado?
- que metodo es mas facil de explicar visualmente?

## Cierre docente

BPE y WordPiece no son importantes por el nombre, sino porque muestran una idea central: el modelo no ve texto como nosotros. Ve piezas, y la forma de partir esas piezas afecta costo, cobertura y calidad.
